# 02 — Feature Engineering for ESP Predictive Maintenance

This notebook demonstrates the domain-specific feature engineering pipeline for Electric Submersible Pump (ESP) sensor data.

**Features created:**
1. Rolling statistics (mean, std, min, max, kurtosis, skewness)
2. Rate-of-change features (first & second derivatives)
3. FFT-based spectral features from vibration signals
4. Cross-sensor interaction features
5. Pump curve deviation features (Affinity Laws)

Run this notebook after downloading data with `python scripts/download_data.py --dataset synthetic`.

In [ ]:
# Install dependencies if running in Colab
# !pip install numpy pandas matplotlib seaborn scipy scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

from src.data.synthetic_generator import generate_esp_dataset, SYNTHETIC_SENSOR_COLS
from src.data.feature_engineering import engineer_features

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 100

## 1. Generate Synthetic Data

We use the physics-based synthetic generator to create a well with a known failure mode.

In [ ]:
# Generate a single well with gas locking failure
df = generate_esp_dataset(
    n_wells=1, timesteps_per_well=1000,
    failure_prob=1.0, random_seed=42,
)

# Force gas locking mode for demonstration
from src.data.synthetic_generator import _inject_gas_locking
rng = np.random.default_rng(42)
failure_start = 600
signals = {col: df[col].values.copy() for col in SYNTHETIC_SENSOR_COLS}
curr, temp, ip, flow = _inject_gas_locking(
    signals['motor_current_A'], signals['motor_temperature_C'],
    signals['intake_pressure_psi'], signals['flow_rate_bpd'],
    failure_start, len(df), rng
)
df['motor_current_A'] = curr
df['motor_temperature_C'] = temp
df['intake_pressure_psi'] = ip
df['flow_rate_bpd'] = flow
df['machine_status'] = np.where(df.index >= failure_start, 'BROKEN', 'NORMAL')

print(f"Dataset shape: {df.shape}")
print(f"Failure starts at timestep: {failure_start}")
print(f"Failure mode: {df['failure_mode'].iloc[0]}")

In [ ]:
# Plot key sensors with failure region
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
t = np.arange(len(df))

for ax, col in zip(axes, ['motor_current_A', 'motor_temperature_C', 'vibration_x_g', 'flow_rate_bpd']):
    ax.plot(t, df[col], linewidth=0.8, color='#378ADD')
    ax.axvspan(failure_start, len(df), color='#E24B4A', alpha=0.15, label='Failure')
    ax.set_ylabel(col, fontsize=9)
    ax.legend(fontsize=8)

axes[-1].set_xlabel('Timestep')
plt.suptitle('ESP Sensor Signals — Gas Locking Failure', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Feature Engineering Pipeline

Apply the full domain-specific feature engineering pipeline.

In [ ]:
df_engineered = engineer_features(
    df,
    sensor_cols=SYNTHETIC_SENSOR_COLS,
    freq_hz=1.0,
    rolling_windows=[10, 30, 60],
    fft_n_components=5,
    include_cross_sensor=True,
)

print(f"Original features: {len(SYNTHETIC_SENSOR_COLS)}")
print(f"Engineered features: {len(df_engineered.columns)}")
print(f"New features added: {len(df_engineered.columns) - len(SYNTHETIC_SENSOR_COLS)}")

In [ ]:
# Show feature categories
original = set(SYNTHETIC_SENSOR_COLS)
all_features = set(df_engineered.columns)
new_features = all_features - original

categories = {
    'Rolling stats': [f for f in new_features if 'rmean' in f or 'rstd' in f or 'rmin' in f],
    'Rate of change': [f for f in new_features if 'diff1' in f or 'diff2' in f or 'ewm' in f],
    'Spectral (FFT)': [f for f in new_features if 'fft' in f or 'spectral' in f],
    'Cross-sensor': [f for f in new_features if 'power' in f or 'thermal' in f or 'diff_' in f or 'coupling' in f],
}

for cat, feats in categories.items():
    print(f"\n{cat}: {len(feats)} features")
    if feats:
        print(f"  Examples: {feats[:3]}")

## 3. Feature Importance Analysis

Which engineered features best distinguish failure from normal operation?

In [ ]:
# Simple feature importance: mutual information with failure label
from sklearn.feature_selection import mutual_info_classif

df['failure'] = (df['machine_status'] == 'BROKEN').astype(int)

# Sample for faster computation
sample_idx = np.random.choice(len(df_engineered), size=min(2000, len(df_engineered)), replace=False)
X = df_engineered.iloc[sample_idx].values
y = df['failure'].iloc[sample_idx].values

# Replace any remaining NaN/inf
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

mi_scores = mutual_info_classif(X, y, random_state=42)
feature_names = list(df_engineered.columns)

# Top 20 features
top_idx = np.argsort(mi_scores)[::-1][:20]
top_features = [feature_names[i] for i in top_idx]
top_scores = mi_scores[top_idx]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_features[::-1], top_scores[::-1], color='#378ADD')
ax.set_xlabel('Mutual Information with Failure Label', fontsize=11)
ax.set_title('Top 20 Most Informative Features', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Rolling Statistics Demonstration

Show how rolling statistics capture degradation patterns.

In [ ]:
# Rolling std of motor current — increases during gas locking
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(t, df['motor_current_A'], linewidth=1, color='#378ADD')
axes[0].axvspan(failure_start, len(df), color='#E24B4A', alpha=0.15)
axes[0].set_ylabel('Motor Current (A)')
axes[0].set_title('Raw Signal')

axes[1].plot(t, df_engineered['motor_current_A_rstd_30'], linewidth=1, color='#EF9F27')
axes[1].axvspan(failure_start, len(df), color='#E24B4A', alpha=0.15)
axes[1].set_ylabel('Rolling Std (window=30)')
axes[1].set_title('Rolling Volatility')
axes[1].set_xlabel('Timestep')

plt.tight_layout()
plt.show()

## 5. Summary

Feature engineering expands 10 raw sensors into 100+ informative features:

| Category | Count | Purpose |
|----------|-------|---------|
| Raw sensors | 10 | Motor current, temperature, pressure, vibration, flow |
| Rolling stats | 60 | Mean, std, min, max, kurtosis, skewness × 3 windows |
| Rate of change | 30 | First/second derivatives, EWM trends |
| Spectral | ~50 | FFT magnitudes, spectral centroid, entropy |
| Cross-sensor | ~5 | Power proxy, thermal load, differential pressure |

**Next step:** Train the LSTM Autoencoder (notebook `03_LSTM_Autoencoder.ipynb`) on these features.